# Pick coordinates from any image

Make sure you have an image of which you know the extents.

## Pick point on a map

One way to get it, is to load an image in GoogleEarth:

**In GoogleEarth app do**

add>image_overlay

**load the image and match the image with the GoogleEarth background**

Take the WGS84 coordinates of the window (location).

**Convert them to your reference system**
Put the convered ponts in an extent tuple or extent array:

extent=(xmin, xmax, ymin, ymax)

**Use ImagePicker to Load the image and start clicking your points**

pnts = ImagePicker(image_file_name)

pick the points with the mouse and finalley press enter to stop and cet all clicked points.

## Pick points in a graph

In this case cut out the graph so you know its extent.
Use this extent with the ImagePickler and pick the points from the figure.

You can also use a digitizer app on the internet, which will be much more advanced
and can get entire graphs based on their color.

e.g. https://plotdigitizer.com

which used to be free but now wants some money ($40,- (2026)).

Most famous sofware to do this:
webplotdigitizer https://apps.automeris.io/wpd4/ which is free

From the same person:
https://web.eecs.utk.edu/~dcostine/personal/PowerDeviceLib/DigiTest/index.html

20 minute instruction video
https://www.youtube.com/watch?v=YfomBJHU9fc





**Make sure the correct backend is used that allows interactive use of figure in a notebook**

With this backend, you only see the image after plt.show() and you must close it to continue.

plt.close('all') may work.

In [1]:
import matplotlib
matplotlib.use("QtAgg")

import matplotlib.pyplot as plt
print(matplotlib.get_backend())

QtAgg


**Imports**

In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from typing import Any
from tools.etc import pickleto, picklefrom, descr

NOTEBOOK_NAME = "impage_picker.ipynb"
print(f"NOTEBOOK_NAME = '{NOTEBOOK_NAME}'")

matplotlib.use("QtAgg")
print(matplotlib.get_backend())

# --- project directory namespace
class Dirs:
    def __init__(self):
        self.rws = "/Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/"
        self.home = os.path.join(self.rws, "src")
        self.images = os.path.join(self.home, '../images')
        self.data = os.path.join(self.home, '../data')
        
dirs = Dirs()

print("Project directory namespace:")
descr(dirs)

NOTEBOOK_NAME = 'impage_picker.ipynb'
QtAgg
Project directory namespace:
rws                  /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/
home                 /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src
images               /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../images
data                 /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data


# Image Pickler

In [3]:
class ImagePicker:
    def __init__(self, image, extent):
        self.image = image
        self.extent = extent

    def pick_points(self, n=1, zoom=False, title="Click points"):

        plt.close('all')

        fig, ax = plt.subplots()

        ax.imshow(
            self.image,
            extent=self.extent,
            origin='upper'
        )

        ax.set_title(title + "\nClick points, ENTER to finish")

        if zoom:
            ax.axis('on')
        else:
            ax.axis('off')

        pts = []

        def onclick(event):
            if event.inaxes != ax:
                return

            if len(pts) >= n:
                return

            x, y = event.xdata, event.ydata
            pts.append((x, y))

            # immediate visual feedback
            ax.plot(x, y, 'r+', markersize=12, mew=2)

            fig.canvas.draw()

        def onkey(event):
            if event.key == 'enter':
                plt.close(fig)

        cid_click = fig.canvas.mpl_connect('button_press_event', onclick)
        cid_key = fig.canvas.mpl_connect('key_press_event', onkey)

        plt.show()

        # clean up connections
        fig.canvas.mpl_disconnect(cid_click)
        fig.canvas.mpl_disconnect(cid_key)

        return pts

# Example two images that were cutout from reports

In [4]:
image_wellen = os.path.join(dirs.images,
        "wellen_Beemster_22_fig3_1_128499_131432_473951_476274.png")
image_raaien = os.path.join(dirs.images,
        "raaien_fig2_1_systeemanalyse_Deltares_128127_130749_471773_475601.png")

for image in [image_wellen, image_raaien]:
    assert os.path.isfile(image), f"FileNotFound {image}"

fig, (ax1, ax2) = plt.subplots(1,2)

img_wellen = Image.open(image_wellen)
img_raaien = Image.open(image_raaien)
ax1.imshow(img_wellen)
ax2.imshow(img_raaien)
ax1.set_title("Wellen")
ax2.set_title("Raaien")

plt.show()

## Extent of the cutout of fig. 3.1 from Beemster et all to pick the wellen

In [7]:
from pyproj import Transformer

# --- transformer from WGS84 -> RD New
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:28992",
    always_xy=True
)

# === Geolocating cut-out of fig 3.1 in Beemster (2022)
# --- NW SE (UL, LR) WGS84 (EPSG:4326)coordinates of the image after matchting the image in GoogleEarth
lonlat = np.array([[4.998927,  52.273840],
                   [5.042055,  52.253094]]
)

# --- Convert them to Amserfoort (EPSG:28992)
print("image_wellen UL, LR")
for lon, lat in lonlat:
    x, y = transformer.transform(lon, lat)
    print(np.round(x), np.round(y))

# Put the results into an extent
extent_wellen=np.array([128499., 131432., 473951., 476274.])
# --- You may add the extent to the file name for later reference:
# --- "wellen_Beemster_22_fig3_1_128499_131432_473951_476274.png"
print(f"\nwellen_extent = {extent_wellen}")

image_wellen UL, LR
128499.0 476274.0
131432.0 473951.0

wellen_extent = [128499. 131432. 473951. 476274.]


In [8]:
# === Geolocating cut-out of fig.2.1 in Systeemanalyse Deltares
# --- NW SE (UL, LR) WGS84 (EPSG:4326) coordinates of image in GoogleEarth
lonlat = np.array([[4.993524,  52.267770],
                   [5.032213,  52.233491]]
)

print("image_raaien UL LR")
for lon, lat in lonlat:
    x, y = transformer.transform(lon, lat)
    print(np.round(x), np.round(y))
    
# --- Resulting extent from geolocating image
extent_raaien = (128127, 130749, 471773, 475601)
# --- Extent added to the file name for later reference
# --- "raaien_fig2_1_systeemanalyse_Deltares_128127_130749_471773_475601.png"
print(f"\nraaien extent = {extent_raaien}")


image_raaien UL LR
128127.0 475601.0
130749.0 471773.0

raaien extent = (128127, 130749, 471773, 475601)


## Pick the wellen from the image

With ImagePicker passing filename and extent, the
image fires up and you can start clicking.
The output is the points as a list of (x,y) tuples
in the correct coordinate system.

In [9]:
if False:
    impckr = ImagePicker(img_wellen, extent=extent_wellen)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()
    print("Clicked points")
    print(pnts)

**Don't forget to store the points, for instance, by pickling them.***

In [10]:
# pickleto(pnts, os.path.join(dirs.data, 'wellen.pkl'))

**Later on, the points can be retrieved by unpickling them**

In [11]:
fig, ax = plt.subplots()
ax.imshow(img_wellen, extent=extent_wellen, origin='upper')

pnts = np.array(picklefrom(os.path.join(dirs.data, 'wellen.pkl')))
for pnt in pnts:
    ax.plot(*pnt, 'ro')
    
plt.show()

Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/wellen.pkl


## Pickle raaien

In [ ]:
if False:
    impckr = ImagePicker(img_raaien, extent=extent_raaien)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()    
    print("Clicked points")
    print(pnts)    


In [23]:
# pickleto(pnts, os.path.join(dirs.data, 'raaien.pkl'))

In [12]:
fig, ax = plt.subplots()
ax.imshow(img_raaien, extent=extent_raaien, origin='upper')

pnts = np.array(picklefrom(os.path.join(dirs.data, 'raaien.pkl')))
for pnt in pnts:
    ax.plot(*pnt, 'ro')
    
plt.show()


Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/raaien.pkl


## Heart line ARK

In [ ]:
if False:
    impckr = ImagePicker(img_raaien, extent=extent_raaien)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()    
    print("Clicked points")
    print(pnts)    

Clicked points
[[129139. 471848.]
 [129779. 475561.]]


In [21]:
ARK_heart_line = np.array([
            [129779., 475561.],
            [129139., 471848.]
])
            
ARK_center_line_GE = np.array([
            [129810.0, 475695.0],
            [129006.0, 471010.0]
             ])

## Unpickle the stored data and plot them with the the two heartlines and the image

It shows that both heartlines are almost identical even though that of Google Earth directly should be more accurate.

In [22]:
wellen = picklefrom(os.path.join(dirs.data, "wellen.pkl"))
raaien = picklefrom(os.path.join(dirs.data, "raaien.pkl"))

fig, ax = plt.subplots()
ax.plot(*ARK_heart_line.T, label="heart_line ARK (Image)")
ax.plot(*ARK_center_line_GE.T, label="center_line_ARK (GE)")
ax.plot(*wellen.T, 'bo', label="wellen")
ax.plot(*raaien.T, 'ro', label='raaien')

if False:
    image = "wellen_Beemster_22_fig3_1_128499_131432_473951_476274.png"
    extent = extent_wellen
else:
    image = "raaien_fig2_1_systeemanalyse_Deltares_128127_130749_471773_475601.png"
    extent = extent_raaien

img = Image.open(os.path.join(dirs.images, image))
#ax.imshow(img, extent=extent_wellen, origin='upper')
ax.imshow(img, extent=extent, origin='upper')
ax.legend()
plt.show()

Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/wellen.pkl
Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/raaien.pkl


In [36]:
def point_line_distance(P, A, B):
    """
    Distance from points P to infinite line through A and B.

    Parameters
    ----------
    P : (..., 2) array_like
        One or more points.
    A, B : (2,) array_like
        Two points defining the line.

    Returns
    -------
    d : ndarray
        Distances.
    """

    P = np.asarray(P)
    A = np.asarray(A)
    B = np.asarray(B)

    AB = B - A
    AP = P - A

    cross = AB[0] * AP[..., 1] - AB[1] * AP[..., 0]

    return np.round(np.abs(cross) / np.linalg.norm(AB), 2), np.sign(cross)

    
# --- Afstand wellen tot ARK center line
A, B = ARK_center_line_GE
wellen = np.array(picklefrom(os.path.join(dirs.data, 'wellen.pkl')))
raaien = np.array(picklefrom(os.path.join(dirs.data, 'raaien.pkl')))

d_wellen, sign_wellen = point_line_distance(wellen, A, B)
print("\nDistance wellen to center line of ARK [m]")
print(d_wellen)
d_raaien, sign_raaien = point_line_distance(raaien, A, B)
print("\nDistance obs. wells to center line of ARK [m]")
print(d_raaien)

fig, ax = plt.subplots()

ax.plot(d_wellen * sign_wellen, np.zeros_like(d_wellen), 'bo', label='wellen')
ax.plot(d_raaien * sign_raaien, np.zeros_like(d_raaien), 'ro', label='buizen')

ax.set_title("Aftand wellen en peilbuizen tot hartlijen of ARK")
ax.grid()
ax.legend()

plt.show()


Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/wellen.pkl
Loaded <function basename at 0x10d1379c0> <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data/raaien.pkl

Distance wellen to center line of ARK [m]
[486.07 468.51 266.23 265.95 287.79 453.55 501.33 549.46 651.54 665.75
 656.79 675.17 538.8  529.32 417.06 378.84 382.04 338.94 292.48 226.47
 654.95 578.3  507.59 478.39 445.05 408.27 345.05 341.18 234.81 239.26
  81.25  80.7  398.25 330.33 270.72 207.93 154.79 130.63  59.94 268.82
 263.29 260.83 220.03 146.46  72.74  99.64  77.88 138.34 165.31 188.08
 243.11 252.14 213.78 255.03 292.57 298.36 247.73 205.63 171.59 156.37
 207.57 231.94 230.1  220.97 440.2  472.3  952.33 722.4  182.6  175.02
 176.81 172.13 169.61 171.35 164.53 153.47 145.43 138.83 136.72  70.46
 145.87 128.12 169.   268.18 310.93 297.62 100.83 125.24 158.04 117.14
 111.8  201.57 103.59 146.59 126.38 256.48 272.5  